# 🔗 Bloque 3 — Sesión 2: Cramér's V, significación estadística y selección de variables

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **3.4** Cramér's V: asociación entre variables categóricas
- **3.5** Significación estadística: p-valor, nivel de confianza e intervalo de confianza
- **3.6** Selección de variables: matrices de correlación, multicolinealidad y criterios de descarte

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación basadas en cálculos estadísticos para detectar la cobertura y el sesgo.  
> **(CE c)** Se ha documentado la descripción del resultado de las verificaciones para que se tomen decisiones de diseño sobre la selección y uso del conjunto de datos en el modelado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, chi2_contingency
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 3.4 Cramér's V: asociación entre variables categóricas

Pearson y Spearman miden relaciones entre variables numéricas. Para cuantificar la **asociación entre dos variables categóricas** necesitamos otra medida: **Cramér's V**.

### Fundamento: el test chi-cuadrado (χ²)

Cramér's V se basa en el estadístico chi-cuadrado, que compara la **tabla de contingencia observada** con la que esperaríamos si las variables fueran independientes:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

El problema del χ² es que su valor depende del tamaño muestral y del número de categorías, por lo que no es directamente comparable entre variables. Cramér's V normaliza el χ² para obtener un valor entre 0 y 1:

$$V = \sqrt{\frac{\chi^2 / n}{\min(k-1, r-1)}}$$

donde $n$ es el tamaño muestral, $k$ el número de columnas y $r$ el de filas de la tabla de contingencia.

| Valor de V | Interpretación |
|-----------|---------------|
| 0.00 – 0.10 | Asociación muy débil o nula |
| 0.10 – 0.30 | Asociación débil |
| 0.30 – 0.50 | Asociación moderada |
| 0.50 – 1.00 | Asociación fuerte |
| 1.00 | Asociación perfecta |

In [ ]:
def cramers_v(col1, col2):
    """Calcula el coeficiente V de Cramér entre dos variables categóricas."""
    tabla = pd.crosstab(col1, col2)
    chi2, p, dof, _ = chi2_contingency(tabla)
    n = tabla.sum().sum()
    v = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
    return v, p

# Ejemplo: asociación entre variables categóricas del Titanic
pares = [
    ('sex', 'survived'),
    ('pclass', 'survived'),
    ('embarked', 'survived'),
    ('sex', 'pclass'),
    ('embarked', 'pclass'),
]

print(f"{'Par de variables':<35} {'Cramér V':>10} {'p-valor':>12} {'Asociación':>20}")
print('─' * 80)
for c1, c2 in pares:
    df_tmp = titanic[[c1, c2]].dropna()
    V, p = cramers_v(df_tmp[c1], df_tmp[c2])
    if V < 0.10: etiqueta = 'Muy débil'
    elif V < 0.30: etiqueta = 'Débil'
    elif V < 0.50: etiqueta = 'Moderada'
    else: etiqueta = '⚠️ Fuerte'
    print(f"{c1} vs {c2:<25} {V:>10.4f} {p:>12.6f} {etiqueta:>20}")

In [ ]:
# Tabla de contingencia visual: sexo vs supervivencia
tabla_sv = pd.crosstab(titanic['sex'], titanic['survived'],
                        rownames=['Sexo'], colnames=['Sobrevivió'])
tabla_sv.columns = ['No', 'Sí']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Frecuencias absolutas
tabla_sv.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'],
              edgecolor='white', width=0.6)
axes[0].set_title('Frecuencias absolutas')
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Nº pasajeros')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Sobrevivió')

# Proporciones (más informativas para comparar)
tabla_pct = tabla_sv.div(tabla_sv.sum(axis=1), axis=0)
tabla_pct.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'],
               edgecolor='white', width=0.6)
axes[1].set_title('Proporciones (por fila)')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Proporción')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sobrevivió')

V, p = cramers_v(titanic['sex'].dropna(), titanic['survived'].dropna())
plt.suptitle(f'Sexo vs. Supervivencia — Cramér V={V:.3f}, p={p:.2e}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz completa de Cramér's V entre todas las categóricas
def matriz_cramers_v(df, columnas_cat):
    """Calcula la matriz de Cramér's V para un conjunto de variables categóricas."""
    n = len(columnas_cat)
    matrix = pd.DataFrame(np.zeros((n, n)), index=columnas_cat, columns=columnas_cat)
    for i, c1 in enumerate(columnas_cat):
        for j, c2 in enumerate(columnas_cat):
            if i == j:
                matrix.loc[c1, c2] = 1.0
            elif i < j:
                df_tmp = df[[c1, c2]].dropna()
                V, _ = cramers_v(df_tmp[c1], df_tmp[c2])
                matrix.loc[c1, c2] = V
                matrix.loc[c2, c1] = V
    return matrix

cols_cat = ['sex', 'pclass', 'embarked', 'who', 'class', 'survived']
mv = matriz_cramers_v(titanic, cols_cat)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(mv, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, linecolor='white',
            ax=ax, square=True)
ax.set_title("Matriz de Cramér's V — Variables categóricas Titanic", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3.5 Significación estadística: p-valor, nivel de confianza e intervalo de confianza

Cuando calculamos una correlación o cualquier estadístico a partir de una muestra, siempre existe la posibilidad de que el resultado sea fruto del azar. La **significación estadística** nos ayuda a cuantificar esa posibilidad.

### El p-valor

El **p-valor** responde a la pregunta: *si en la población real no hubiera relación (H₀), ¿cuál sería la probabilidad de observar una correlación al menos tan extrema como la que hemos medido?*

- **p-valor pequeño (< α):** resultado poco probable bajo H₀ → rechazamos H₀ → resultado **estadísticamente significativo**
- **p-valor grande (≥ α):** resultado compatible con H₀ → no podemos rechazarla

El umbral habitual es **α = 0.05** (5% de probabilidad de error de tipo I).

> ⚠️ **Errores frecuentes de interpretación:**  
> – El p-valor NO es la probabilidad de que H₀ sea verdadera.  
> – Significativo ≠ importante. Con n grande, correlaciones tiny (r=0.01) pueden ser "significativas".  
> – No significativo ≠ sin efecto. Con n pequeño, correlaciones reales pueden no detectarse.

In [ ]:
# Pearson con p-valor
cols_num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
print("Correlaciones de Pearson con p-valor (Penguins):
")
print(f"{'Par':<45} {'r':>8} {'p-valor':>12} {'Significativa (α=0.05)?':>25}")
print('─' * 93)

for i, c1 in enumerate(cols_num):
    for j, c2 in enumerate(cols_num):
        if j <= i: continue
        r, p = pearsonr(penguins[c1], penguins[c2])
        sig = '✅ Sí' if p < 0.05 else '❌ No'
        print(f"{c1} vs {c2:<30} {r:>8.4f} {p:>12.6f} {sig:>25}")

In [ ]:
# Efecto del tamaño muestral en la significación
# La misma correlación puede ser significativa o no según n
r_real = 0.15  # correlación pequeña pero real

resultados = []
for n in [20, 50, 100, 200, 500, 1000, 5000]:
    x = np.random.normal(0, 1, n)
    y = r_real * x + np.sqrt(1 - r_real**2) * np.random.normal(0, 1, n)
    r_obs, p = pearsonr(x, y)
    resultados.append({'n': n, 'r observado': round(r_obs, 4),
                        'p-valor': round(p, 5), 'Significativa': p < 0.05})

df_res = pd.DataFrame(resultados).set_index('n')
print(f"Correlación real en la población: r = {r_real}")
print()
print(df_res.to_string())
print()
print("→ La misma correlación de r≈0.15 pasa de no significativa a muy significativa")
print("  simplemente aumentando el tamaño de la muestra.")

### Intervalo de confianza para la correlación

El coeficiente r es un estimador puntual: calculado sobre una muestra diferente, daría un valor distinto. El **intervalo de confianza** (IC) indica el rango dentro del cual cae el verdadero parámetro poblacional con una probabilidad dada (habitualmente el 95%).

Para la correlación de Pearson se usa la **transformación z de Fisher** para construir el IC:

$$z = \frac{1}{2} \ln\left(\frac{1+r}{1-r}\right), \quad SE_z = \frac{1}{\sqrt{n-3}}$$

In [ ]:
def ic_pearson(r, n, alpha=0.05):
    """Calcula el intervalo de confianza para la correlación de Pearson mediante la z de Fisher."""
    z = np.arctanh(r)          # transformación z de Fisher
    se = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    z_lo = z - z_crit * se
    z_hi = z + z_crit * se
    r_lo = np.tanh(z_lo)       # transformación inversa
    r_hi = np.tanh(z_hi)
    return r_lo, r_hi

# IC para distintos tamaños de muestra con la misma r
r_obs = 0.60
print(f"IC al 95% para r = {r_obs}:
")
print(f"{'n':>6}  {'IC inferior':>12}  {'IC superior':>12}  {'Amplitud':>10}")
print('─' * 45)
for n in [20, 50, 100, 300, 1000]:
    lo, hi = ic_pearson(r_obs, n)
    print(f"{n:>6}  {lo:>12.4f}  {hi:>12.4f}  {hi-lo:>10.4f}")

print()
print("→ Con n=20, el IC es amplísimo: r=0.60 podría ser desde 0.19 hasta 0.83.")
print("  Con n=1000, el IC es preciso: sabemos que r está cerca de 0.60.")

In [ ]:
# Visualización del IC para distintas correlaciones observadas (n=100)
n = 100
r_vals = np.linspace(-0.9, 0.9, 37)
fig, ax = plt.subplots(figsize=(11, 5))

for r_v in r_vals:
    lo, hi = ic_pearson(r_v, n)
    color = 'steelblue' if r_v >= 0 else 'coral'
    ax.plot([r_v, r_v], [lo, hi], color=color, linewidth=1.5, alpha=0.7)
    ax.scatter([r_v], [r_v], color=color, s=20, zorder=3)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('r observado')
ax.set_ylabel('Intervalo de confianza (95%)')
ax.set_title(f'IC al 95% para la correlación de Pearson (n={n})', fontsize=13)
ax.plot([], [], color='steelblue', label='r positiva')
ax.plot([], [], color='coral', label='r negativa')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3.6 Selección de variables mediante correlación

Una de las aplicaciones más directas de los análisis de correlación en ML es la **selección de variables** (feature selection). El objetivo es identificar qué atributos son más relevantes para predecir la variable objetivo y cuáles son redundantes o irrelevantes.

### Multicolinealidad

Cuando dos o más variables predictoras están altamente correlacionadas entre sí, tenemos **multicolinealidad**. Esto causa problemas en:
- Regresión lineal/logística: los coeficientes se vuelven inestables
- Interpretabilidad: no podemos distinguir el efecto de cada variable
- Redundancia: aportan información muy similar al modelo

**Criterio habitual:** si |r| > 0.85 entre dos predictores, considerar eliminar uno.

In [ ]:
# Detección de multicolinealidad
cols_pred = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
r_mat = penguins[cols_pred].corr()

print("Pares con correlación alta (|r| > 0.7):")
print()
for i, c1 in enumerate(cols_pred):
    for j, c2 in enumerate(cols_pred):
        if j <= i: continue
        r = r_mat.loc[c1, c2]
        if abs(r) > 0.7:
            nivel = '⚠️ Multicolinealidad severa' if abs(r) > 0.85 else '〰️ Correlación alta'
            print(f"  {c1} vs {c2}: r = {r:.3f}  {nivel}")

In [ ]:
# Función: informe de selección de variables por correlación
def informe_seleccion_variables(df, cols_predictoras, col_objetivo=None,
                                 umbral_multicolinealidad=0.85,
                                 umbral_relevancia=0.10):
    """
    Genera un informe de selección de variables basado en:
    - Correlación con la variable objetivo (relevancia)
    - Correlación entre predictores (multicolinealidad)
    """
    datos = df[cols_predictoras + ([col_objetivo] if col_objetivo else [])].dropna()
    
    print("=" * 65)
    print("INFORME DE SELECCIÓN DE VARIABLES")
    print("=" * 65)
    
    # 1. Relevancia respecto al objetivo
    if col_objetivo:
        print(f"\n1. CORRELACIÓN CON '{col_objetivo}':")
        print(f"   (umbral mínimo de relevancia: |r| > {umbral_relevancia})")
        print()
        relevancia = []
        for col in cols_predictoras:
            try:
                r, p = pearsonr(datos[col], datos[col_objetivo])
            except Exception:
                r, p = 0, 1
            relevante = '✅' if abs(r) > umbral_relevancia and p < 0.05 else '❌ Baja'
            relevancia.append((col, r, p, relevante))
            print(f"   {col:<25} r={r:+.3f}  p={p:.4f}  {relevante}")
        
        print()
        print("   Variables con baja relevancia (candidatas a descartar):")
        for col, r, p, rel in relevancia:
            if '❌' in rel:
                print(f"   → {col}")
    
    # 2. Multicolinealidad
    print(f"\n2. MULTICOLINEALIDAD (umbral: |r| > {umbral_multicolinealidad}):")
    r_pred = datos[cols_predictoras].corr()
    hay_multi = False
    for i, c1 in enumerate(cols_predictoras):
        for j, c2 in enumerate(cols_predictoras):
            if j <= i: continue
            r = r_pred.loc[c1, c2]
            if abs(r) > umbral_multicolinealidad:
                print(f"   ⚠️  {c1} vs {c2}: r = {r:.3f} → considera eliminar una")
                hay_multi = True
    if not hay_multi:
        print("   ✅ No se detecta multicolinealidad severa.")
    
    print("\n" + "=" * 65)

# Aplicación al dataset Penguins (objetivo: body_mass_g)
cols_pred_p = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
informe_seleccion_variables(penguins, cols_pred_p, col_objetivo='body_mass_g')

In [ ]:
# Visualización integrada: relevancia vs multicolinealidad
cols_titanic = ['age', 'fare', 'sibsp', 'parch', 'pclass']
objetivo = 'survived'

df_t = titanic[cols_titanic + [objetivo]].dropna()
df_t['pclass'] = df_t['pclass'].astype(float)

r_obj = df_t[cols_titanic].corrwith(df_t[objetivo])
r_pred = df_t[cols_titanic].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de relevancia
colores = ['steelblue' if abs(v) > 0.1 else 'lightgray' for v in r_obj]
axes[0].barh(r_obj.index, r_obj.values, color=colores, edgecolor='white')
axes[0].axvline(0.1, color='crimson', linestyle='--', linewidth=1.5, label='Umbral +0.1')
axes[0].axvline(-0.1, color='crimson', linestyle='--', linewidth=1.5, label='Umbral -0.1')
axes[0].set_title(f"Correlación con '{objetivo}'", fontsize=12)
axes[0].set_xlabel('r de Pearson')
axes[0].legend(fontsize=9)

# Heatmap de multicolinealidad
sns.heatmap(r_pred, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            ax=axes[1], square=True)
axes[1].set_title('Multicolinealidad entre predictores', fontsize=12)

plt.suptitle('Análisis de selección de variables — Titanic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Informe completo de selección para Titanic
informe_seleccion_variables(
    df_t,
    cols_titanic,
    col_objetivo='survived',
    umbral_multicolinealidad=0.85,
    umbral_relevancia=0.10
)

---
## 🔧 Ejercicios

### Ejercicio 1 — Cramér's V en Diamonds

El dataset `diamonds` tiene varias variables categóricas: `cut`, `color` y `clarity`. Calcula la matriz de Cramér's V entre ellas. ¿Cuáles están más asociadas? ¿Tendría sentido incluir todas en un modelo o hay redundancia?

In [ ]:
# Aplica matriz_cramers_v() al dataset diamonds
# cols_cat_d = ['cut', 'color', 'clarity']
# mv_d = matriz_cramers_v(diamonds, cols_cat_d)
# print(mv_d.round(3))

# Visualiza el heatmap
# fig, ax = plt.subplots(figsize=(6, 5))
# sns.heatmap(mv_d, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, vmax=1, ax=ax)
# ...

# ¿Qué par tiene asociación más fuerte? ¿Lo eliminarías?


### Ejercicio 2 — IC de correlación

Para el par `flipper_length_mm` vs `body_mass_g` del dataset `penguins`:
1. Calcula r y su IC al 95%
2. ¿El IC excluye el 0? ¿Qué implica eso?
3. Repite el cálculo con solo las primeras 20 filas del dataset. ¿Cómo cambia el IC?

In [ ]:
# 1. IC con todos los datos
# peng_full = penguins[['flipper_length_mm', 'body_mass_g']].dropna()
# r_full, p_full = pearsonr(peng_full['flipper_length_mm'], peng_full['body_mass_g'])
# lo_full, hi_full = ic_pearson(r_full, len(peng_full))
# print(f"n={len(peng_full)}: r={r_full:.4f}  IC=[{lo_full:.4f}, {hi_full:.4f}]")

# 2. IC con solo 20 observaciones
# peng_20 = peng_full.head(20)
# r_20, _ = pearsonr(peng_20['flipper_length_mm'], peng_20['body_mass_g'])
# lo_20, hi_20 = ic_pearson(r_20, 20)
# print(f"n=20: r={r_20:.4f}  IC=[{lo_20:.4f}, {hi_20:.4f}]")

# ¿Qué conclusiones extraes sobre el efecto del tamaño muestral?


### Ejercicio 3 — Informe de selección completo

Aplica `informe_seleccion_variables()` al dataset `diamonds` con las variables numéricas `carat`, `depth`, `table`, `x`, `y`, `z` y el objetivo `price`. ¿Qué variables son más relevantes? ¿Hay multicolinealidad entre `x`, `y` y `z`? ¿Qué harías con ellas?

In [ ]:
# cols_pred_d = ['carat', 'depth', 'table', 'x', 'y', 'z']
# informe_seleccion_variables(diamonds, cols_pred_d, col_objetivo='price',
#                              umbral_multicolinealidad=0.85, umbral_relevancia=0.2)

# Después de leer el informe, ¿qué subset de variables usarías para predecir el precio?
# Justifica tu respuesta.


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Cramér's V** | Asociación entre categóricas. Rango [0,1]. Basado en chi-cuadrado. |
| **chi-cuadrado (χ²)** | Contrasta independencia en tabla de contingencia. Sensible a n. |
| **p-valor** | Probabilidad de observar el resultado si H₀ fuera cierta. Umbral habitual: α=0.05. |
| **Significativo ≠ importante** | Con n grande, efectos mínimos pueden ser "significativos". Interpretar siempre junto al tamaño del efecto. |
| **IC de Pearson** | Rango plausible para el r real. Se amplía con n pequeño. Construido con z de Fisher. |
| **Multicolinealidad** | \|r\| > 0.85 entre predictores → redundancia → considerar eliminar uno. |
| **Selección por correlación** | Combinar relevancia (cor. con objetivo) + multicolinealidad para seleccionar atributos. |

### 🔜 Próxima sesión
**Bloque 4 — Sesión 1:** Tests de independencia estadística. Chi-cuadrado, t-test y ANOVA.